# **Silver — Vacinação Dengue (SP)**

Roda **depois** que `fiap.bronze.PNI_VACINACAO_SP` já estiver populada. Gera a Silver `fiap.silver.VACINACAO_DENGUE_SP`, que é o que o `nb_vacinacao_analysis.ipynb` espera encontrar.

In [0]:
from pyspark.sql.functions import col, trim, upper, substring, to_date, coalesce, year, month, current_timestamp, lit

CATALOGO = "fiap"
UF = "SP"

df_bronze = spark.table(f"{CATALOGO}.bronze.PNI_VACINACAO_{UF}")
print(f"Registros na Bronze: {df_bronze.count():,.0f}")


def campo(df, *nomes):
    existentes = [c for c in nomes if c in df.columns]
    if not existentes:
        return lit(None)
    return coalesce(*[col(c) for c in existentes])


df_silver = (
    df_bronze
    .withColumn("cod_municipio_paciente_raw", campo(df_bronze, "codigo_municipio_paciente", "co_municipio_paciente"))
    .withColumn("sigla_vacina_raw", campo(df_bronze, "sigla_vacina", "sg_vacina"))
    .withColumn("data_vacina_raw", campo(df_bronze, "data_vacina", "dt_vacina"))
    .withColumn("idade_paciente_raw", campo(df_bronze, "numero_idade_paciente", "nu_idade_paciente"))
    .withColumn("sexo_paciente_raw", campo(df_bronze, "tipo_sexo_paciente", "tp_sexo_paciente"))
    .withColumn("cnes_estabelecimento_raw", campo(df_bronze, "codigo_cnes_estabelecimento", "co_cnes_estabelecimento"))
    .withColumn("documento_raw", campo(df_bronze, "codigo_documento", "co_documento"))
)

# Tratamento de nulos / inconsistencias:
# 1. Remove registros sem codigo de municipio (nao da pra localizar/cruzar)
# 2. Remove registros sem data de vacinacao (nao da pra agregar por periodo)
# 3. Normaliza texto (trim + upper) para evitar duplicidade por grafia
# 4. Deduplica por documento (RNDS), para nao contar a mesma dose duas vezes
df_silver = (
    df_silver
    .filter(col("cod_municipio_paciente_raw").isNotNull())
    .filter(col("data_vacina_raw").isNotNull())
    .withColumn("cod_ibge_municipio", substring(trim(col("cod_municipio_paciente_raw")), 1, 6))
    .withColumn("sigla_vacina_tratada", upper(trim(col("sigla_vacina_raw"))))
    .withColumn("dt_vacina_tratada", to_date(col("data_vacina_raw")))
    .dropDuplicates(["documento_raw"])
)

# Filtro de dengue mantido aqui tambem por seguranca, mesmo que o streaming ja tenha filtrado
df_silver = df_silver.filter(
    col("sigla_vacina_tratada").contains("DENGUE") | col("sigla_vacina_tratada").contains("QDENGA")
)

df_vacinacao_dengue_sp = df_silver.select(
    col("cod_ibge_municipio"),
    col("sigla_vacina_tratada").alias("sigla_vacina"),
    col("dt_vacina_tratada").alias("dt_vacina"),
    year(col("dt_vacina_tratada")).alias("ano_referencia"),
    month(col("dt_vacina_tratada")).alias("mes_referencia"),
    col("idade_paciente_raw").cast("int").alias("idade_paciente"),
    col("sexo_paciente_raw").alias("sexo_paciente"),
    col("cnes_estabelecimento_raw").alias("cnes_estabelecimento"),
    current_timestamp().alias("data_carga"),
)

count_final = df_vacinacao_dengue_sp.count()
print(f"Registros de vacinacao contra dengue em SP apos tratamento: {count_final:,.0f}")

df_vacinacao_dengue_sp.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.silver.VACINACAO_DENGUE_SP")

print(f"Silver {CATALOGO}.silver.VACINACAO_DENGUE_SP atualizada")

## Conferência rápida

In [0]:
%sql
SELECT ano_referencia, mes_referencia, COUNT(*) AS doses_aplicadas
FROM fiap.silver.VACINACAO_DENGUE_SP
GROUP BY ano_referencia, mes_referencia
ORDER BY ano_referencia, mes_referencia